# DeepForest model training

`src/` recovers bird positions from screenshots where a point-counting tool baked
coloured dots into the pixels. `scripts/export_dataset.py` maps those positions onto
the original photographs and writes a DeepForest dataset. This notebook is the last
step: does a detector trained on them work?

**Needs a GPU.** Every other stage runs on a laptop CPU. A Colab T4 is enough.

**Runs anywhere** - Colab in the browser, a Colab runtime attached to VS Code, or any
GPU machine. Nothing here imports `google.colab`, so no cell depends on the browser UI.

**Reads** `results/dataset/annotations_deepforest.csv` from the repository, and pulls
the 25 photographs it names from the public S3 bucket.

**Reports two numbers, not one:** the pretrained model on held-out frames, then the
fine-tuned model on the same frames. Without the first there is no way to say the
recovered annotations taught it anything.

## 1. GPU

In [ ]:
import torch, warnings
warnings.filterwarnings("ignore")

if not torch.cuda.is_available():
    raise SystemExit(
        "No GPU. In Colab: Runtime -> Change runtime type -> T4 GPU. "
        "In VS Code: pick the Colab GPU runtime as the kernel. "
        "Training on CPU is not worth starting - this is the one stage that needs one.")

print("gpu   :", torch.cuda.get_device_name(0))
print("memory:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
print("torch :", torch.__version__)

## 2. Install

In [ ]:
!pip -q install deepforest==2.1.0
import deepforest
print("deepforest", deepforest.__version__)

## 3. The dataset

Cloned rather than uploaded, so the same cell works in the browser and from VS Code.
The runtime is a different machine from the laptop, so the CSV has to arrive over the
network either way.

In [ ]:
import os, pathlib, urllib.parse, urllib.request
import pandas as pd

REPO = "https://github.com/vickysharma-prog/Recovering-computer-vision-annotations.git"
BRANCH = "feat/map-and-export"

# The branch while the pull request is open, main once it is merged and the
# branch is deleted. Trying both keeps this cell working either side of a merge.
if not os.path.exists("repo"):
    for ref in (BRANCH, "main"):
        rc = os.system(f"git clone --depth 1 --branch {ref} {REPO} repo 2>/dev/null")
        if rc == 0:
            print("cloned", ref)
            break

CSV = pathlib.Path("repo/results/dataset/annotations_deepforest.csv")
assert CSV.exists(), (
    "annotations_deepforest.csv is missing. It is produced by "
    "scripts/export_dataset.py and has to be committed before this "
    "notebook can read it.")

ann = pd.read_csv(CSV)
print(f"{len(ann)} boxes over {ann.image_path.nunique()} frames, "
      f"{ann.label.nunique()} labels")
print(ann.head(3).to_string(index=False))


## 4. The photographs

Public bucket, no credentials. About 90 MB for the 25.

In [ ]:
BUCKET = "https://twi-aviandata.s3.amazonaws.com/avian_monitoring/"
ROOT = pathlib.Path("photos")

paths = sorted(ann.image_path.unique())
for i, rel in enumerate(paths, 1):
    out = ROOT / rel
    if out.exists():
        continue
    out.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(BUCKET + urllib.parse.quote(rel), out)
    print(f"  {i:2d}/{len(paths)}  {os.path.basename(rel)}")

got = list(ROOT.rglob("*.jpg"))
assert len(got) == len(paths), f"only {len(got)} of {len(paths)} photographs arrived"
print(f"\n{len(got)} photographs, {sum(p.stat().st_size for p in got) / 1e6:.0f} MB")

## 5. Box sizes are per frame, and that is deliberate

Each frame's box was measured from that frame's own birds (`src/birdsize.py`). Eleven
years of surveys flew focal lengths from 28mm to 300mm, so a bird spans 10px on one
photograph and 41px on another. One box size would be several times wrong on most of
the set.

In [ ]:
side = (ann.xmax - ann.xmin).round()
by_frame = ann.assign(side=side).groupby("image_path").side.median()

print(f"box side, per box  : {side.min():.0f}-{side.max():.0f}px, "
      f"median {side.median():.0f}px")
print(f"box side, per frame: {by_frame.min():.0f}-{by_frame.max():.0f}px, "
      f"median {by_frame.median():.0f}px")

## 6. Tiles

DeepForest trains on crops, not on 15-megapixel photographs. `split_raster` cuts each
one into overlapping tiles and rewrites the boxes into tile coordinates.

The assertion matters. A box that falls off a tile edge is dropped silently by most
tiling code, and a training set that quietly lost a third of its annotations still
trains and still reports a number.

In [ ]:
import os, time, warnings
warnings.filterwarnings("ignore")
from deepforest.preprocess import split_raster

# One class. The CSV carries 44 species labels and keeps them, but 19 of them hold
# fewer than 80 dots and two hold fewer than three, so a 44-class detector would be
# learning most of its classes from a handful of examples. DeepForest's bird model is
# single-class, and species stay recoverable through frame and legend_row.
train_ann = ann.copy()
train_ann["label"] = "Bird"

PATCH, OVERLAP = 400, 0.15
tiles, t0 = [], time.time()

for n, (rel, g) in enumerate(train_ann.groupby("image_path"), 1):
    g = g.copy()
    g["image_path"] = os.path.basename(rel)
    out = split_raster(annotations_file=g,
                       path_to_raster=str(ROOT / rel),
                       root_dir=str((ROOT / rel).parent),
                       save_dir="tiles",
                       patch_size=PATCH, patch_overlap=OVERLAP)
    out["frame"] = os.path.splitext(os.path.basename(rel))[0]
    tiles.append(out)
    if n % 5 == 0:
        print(f"  {n:2d}/{train_ann.image_path.nunique()}  {time.time()-t0:5.0f}s")

tiled = pd.concat(tiles, ignore_index=True)
ratio = len(tiled) / len(train_ann)
print(f"\n{len(train_ann)} boxes -> {len(tiled)} on {tiled.image_path.nunique()} "
      f"tiles ({ratio:.2f} per original; overlap duplicates some)")

# A box that falls off a tile edge is dropped silently by most tiling code, and a
# training set that quietly lost a third of its annotations still trains and still
# reports a number.
assert ratio > 0.9, f"tiling lost boxes: only {ratio:.2f} survived"
print("assert passed: no boxes lost")


## 7. Split by frame, never by dot

Two dots from the same photograph on either side of the split is not a held-out test:
the model has already seen that background, that colony, that light. Whole frames go
one way or the other.

In [ ]:
import numpy as np

frames = sorted(tiled.frame.unique())
rng = np.random.default_rng(0)
rng.shuffle(frames)
cut = int(0.75 * len(frames))
train_frames, test_frames = set(frames[:cut]), set(frames[cut:])

# split_raster also returns a geometry column. DeepForest reads the six
# columns below, so only those are written.
COLS = ["image_path", "xmin", "ymin", "xmax", "ymax", "label"]
train = tiled[tiled.frame.isin(train_frames)][COLS]
test = tiled[tiled.frame.isin(test_frames)][COLS]
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)

assert not (train_frames & test_frames), "a frame is on both sides"
print(f"train {len(train):6d} boxes on {len(train_frames)} frames")
print(f"test  {len(test):6d} boxes on {len(test_frames)} frames")
print()
print("held out:", ", ".join(sorted(test_frames)))


## 8. Autocast off, and checked

SAM 3 leaves a global bfloat16 autocast enabled that survives deleting the model.
DeepForest then trains without error and returns every score as 1.0. Nothing here runs
SAM 3, but the check costs nothing and the failure is silent.

In [ ]:
torch.set_autocast_enabled(False)
assert not torch.is_autocast_enabled(), "autocast is on; every score would come back 1.0"
print("autocast off")

## 9. The pretrained model, before anything is changed

Measured first, so the comparison has both halves.

In [ ]:
from deepforest import main as df_main


def evaluate(model, csv_path, root="tiles"):
    # root_dir is a keyword here. Passing it positionally lands it in
    # iou_threshold, which fails quietly rather than loudly.
    return model.evaluate(csv_path, root_dir=root, iou_threshold=0.4)


base = df_main.deepforest()
base.load_model("weecology/deepforest-bird")
base.config.score_thresh = 0.3

baseline = evaluate(base, "test.csv")
print("pretrained, on held-out frames")
print(f"  precision {baseline['box_precision']:.3f}")
print(f"  recall    {baseline['box_recall']:.3f}")


## 10. Fine-tune

In [ ]:
model = df_main.deepforest()
model.load_model("weecology/deepforest-bird")

model.config.label_dict = {"Bird": 0}
model.config.num_classes = 1
model.config.train.csv_file = "train.csv"
model.config.train.root_dir = "tiles"
model.config.train.epochs = 15
model.config.train.lr = 0.0001
model.config.batch_size = 4
model.config.score_thresh = 0.3
model.config.validation.csv_file = "test.csv"
model.config.validation.root_dir = "tiles"

model.create_trainer()
model.trainer.fit(model)

tuned = evaluate(model, "test.csv")
print()
print("held-out frames      pretrained -> fine-tuned")
print(f"  precision          {baseline['box_precision']:.3f} -> "
      f"{tuned['box_precision']:.3f}")
print(f"  recall             {baseline['box_recall']:.3f} -> "
      f"{tuned['box_recall']:.3f}")


## 11. Where the errors are

One pair of numbers hides which frames fail. Sorted by box size, because that is the
axis most likely to matter: a 50px box collects several positive anchors where a 16px
box collects one, so recall on the small-bird frames is the first thing expected to
suffer.

In [ ]:
per_frame = []
for f in sorted(test_frames):
    sub = tiled[tiled.frame == f][COLS]
    if sub.empty:
        continue
    sub.to_csv("_one.csv", index=False)
    r = evaluate(model, "_one.csv")
    per_frame.append(dict(frame=f, boxes=len(sub),
                          side=float((sub.xmax - sub.xmin).median()),
                          precision=float(r["box_precision"]),
                          recall=float(r["box_recall"])))

pf = pd.DataFrame(per_frame).sort_values("side")
print(pf.to_string(index=False, float_format=lambda v: f"{v:.3f}"))


## 12. If the small-bird frames lose recall

Two levers, neither applied blind. Run the plain configuration first; reach for these
only if section 11 shows small boxes doing worse.

**Upscale the tiles.** Doubling tile resolution puts every box at or above 32px, the
smallest anchor torchvision's RetinaNet generates. Leaves the architecture and the
pretrained weights untouched; costs four times the pixels. Try this first.

**Smaller anchors.** `deepforest.models.retinanet.Model.create_anchor_generator` exists
for it and is not one line: its own default is a single size tuple against five FPN
levels and raises an assertion, and a correctly shaped version moves anchors per
location from 9 to 21, so the head must be rebuilt and the pretrained head weights no
longer fit.

Neither is required for training to work at all. torchvision's RetinaNet matches with
`allow_low_quality_matches=True`, so every box down to 16px is assigned its best anchor
and none is dropped.

## 13. Save

In [ ]:
import json

model.trainer.save_checkpoint("deepforest_birds_recovered.ckpt")

metrics = dict(
    baseline={k: float(baseline[k]) for k in ("box_precision", "box_recall")},
    finetuned={k: float(tuned[k]) for k in ("box_precision", "box_recall")},
    train_boxes=len(train), test_boxes=len(test),
    train_frames=sorted(train_frames), test_frames=sorted(test_frames),
    patch_size=PATCH, patch_overlap=OVERLAP,
    epochs=model.config.train.epochs, lr=model.config.train.lr,
    per_frame=per_frame)
with open("metrics.json", "w") as fh:
    json.dump(metrics, fh, indent=2)

print("wrote deepforest_birds_recovered.ckpt and metrics.json")
print("copy both off the runtime before it recycles - it does not keep them")